In [209]:
import sys
import os

# Add the path to the parent directory
sys.path.append(os.path.abspath(".."))

# Now you can import the module
from dep.openai_api import OpenAIChatCompletion
from dep.response_formats import SyntheticPositiveFactsheetResponse,AspectQueryResponse

In [20]:
ASPECTS = [
    "Industry", 
    "Customer Segment", 
    "Products & Services", 
    "Business Model", 
    "Technology Used", 
    "Revenue Model"
]

In [24]:
# res=llm.get_text_completion(prompt,0)['choices'][0]['message']['content']
import json
def generate_queries(aspects,n_queries=100,response_format=AspectQueryResponse):
    
    success=False
    
    sys_msg_path1='../prompts/aspect_queries.txt'
    system_message1=open(sys_msg_path1,'r').read()
    llm1=OpenAIChatCompletion(model='gpt-4o',system_message=system_message1)

    if aspects=='Industry':
        print('induystr')
        prompt1=f"Please provide {n_queries} queries for the aspects: {aspects}. Please Provide Industries such that their exist businesses for which these industries are customer industries. Customer industry of a company is the industry to which company sells its solutions. "
    else:
        prompt1=f"Please provide {n_queries} queries for the aspects: {aspects}"
    

    temperature=0.5
    
    res1=llm1.structured_response(prompt1,temperature,response_format)
    try:
        r1=json.loads(res1)
        success=True
        return success,r1
        
    except Exception as e:
        
        print(f'failed for {aspects} - {e}')
        success=False
        return success,{}

In [293]:
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from typing import Callable, List, Any, TypeVar, Optional, Tuple, Dict

T = TypeVar('T')
R = TypeVar('R')

def parallel_execute(
    func: Callable[..., R],
    argument_sets: List[Tuple[List, Dict]],
    max_workers: Optional[int] = None,
    show_progress: bool = True,
    desc: str = "Processing items"
) -> List[R]:
    """
    Execute a function in parallel across multiple threads with different argument sets.
    
    Args:
        func: The function to execute in parallel
        argument_sets: List of (args, kwargs) tuples, where:
            - args is a list of positional arguments
            - kwargs is a dictionary of keyword arguments
        max_workers: Maximum number of threads to use (defaults to min(32, len(argument_sets)))
        show_progress: Whether to show a progress bar
        desc: Description for the progress bar
        
    Returns:
        List of results in the same order as the input argument_sets
    """
    if max_workers is None:
        max_workers = min(1, len(argument_sets))
    
    results = [None] * len(argument_sets)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Define a worker function that captures the index
        def worker(idx, args_list, kwargs_dict):
            return idx, func(*args_list, **kwargs_dict)
        
        # Submit all tasks
        futures = {}
        for idx, (args, kwargs) in enumerate(argument_sets):
            future = executor.submit(worker, idx, args, kwargs)
            futures[future] = idx
        
        # Process results as they complete
        iterator = concurrent.futures.as_completed(futures)
        if show_progress:
            iterator = tqdm(iterator, total=len(futures), desc=desc)
            
        for future in iterator:
            idx, result = future.result()
            results[idx] = result
    
    return results

In [22]:
# and common enough that ColBERT would likely have seen similar terms during pre-training

In [23]:
# generate_queries('Customer Segment')

In [18]:
aspect_queries={}
from tqdm import tqdm
for aspect in tqdm(ASPECTS):
        
    if aspect=='Revenue Model':
        n_queries=25
    elif aspect=='Business Model':
        n_queries=10
    elif aspect=='Technology Used':
        n_queries=30
    else:
        n_queries=100
    queries=generate_queries(aspect,n_queries=n_queries)
    aspect_queries[aspect]=queries

  0%|          | 0/6 [00:00<?, ?it/s]

induystr


100%|██████████| 6/6 [00:38<00:00,  6.35s/it]


In [28]:
!pwd

/home/ec2-user/SageMaker/ColBERT


In [29]:
out_dir='../synthetic_data/complete_aspect_training'
os.makedirs(out_dir)

In [30]:
import pickle
with open('../synthetic_data/complete_aspect_training/aspect_queries.pkl','wb') as f:
    pickle.dump(aspect_queries,f)

In [19]:
aspect_queries

{'Industry': (True,
  {'queries': ['Healthcare',
    'Finance',
    'Retail',
    'Automotive',
    'Technology',
    'Education',
    'Manufacturing',
    'Construction',
    'Real Estate',
    'Energy',
    'Telecommunications',
    'Transportation',
    'Agriculture',
    'Pharmaceutical',
    'Food and Beverage',
    'Hospitality',
    'Entertainment',
    'Media',
    'Aerospace',
    'Defense',
    'Consumer Electronics',
    'Fashion',
    'Beauty',
    'Insurance',
    'Banking',
    'Legal',
    'Logistics',
    'Mining',
    'Chemicals',
    'Biotechnology',
    'Renewable Energy',
    'Utilities',
    'Public Sector',
    'Non-Profit',
    'Sports',
    'Gaming',
    'Textiles',
    'Furniture',
    'Petroleum',
    'Metals',
    'Paper and Pulp',
    'Printing',
    'Marine',
    'Fishing',
    'Forestry',
    'Plastics',
    'Ceramics',
    'Glass',
    'Jewelry',
    'Tobacco',
    'Advertising',
    'Consulting',
    'Event Management',
    'Travel',
    'Tourism',
    '

In [ ]:
Total dataset:
per query 10 positives : For consumer industry and industry: 30 positives
Industry: 
    Negatives from Corresponding Consumer Industry: per positive 10-20 negatives
    Some negatives from people operating in a different industry? - 10 negatives
Customer Industry:
    Negatives from Corresponding Industry: per positive 10-30 negatives
Business Model:
    Synthetic Negatives : 30 per query 
Revenue Model:
    Synthetic Negatives : 30 per wquery 
Technology Used: 
    Synthetic Negatives: 30 per query 
Customer Segment: 
    Choose Other consumer Segments which are likely to be negative 



Cost Estimate:

Input tokens:
60*100*2*512 (Industry+Consumer Industry) + (100*100*3) (Industry intra query negatives) + (500*3*20) (Bm+RM+Tech negatives creation)

Output Tokens:
60*100*2*10 + (100*10) + (20*3*512) 


In [35]:
output_cost=10*(60*100*2*10 + (100*10) + (20*3*512*30))/1000000

In [36]:
output_cost

10.426

In [37]:
dataset_size=300*100*2 + 30*10*20*3 + 300*100

In [38]:
dataset_size

108000

In [39]:
# If we generate positives for Consumer Industry each query

10*(100*30*512/1000000)

15.36

In [40]:
!pwd

/home/ec2-user/SageMaker/ColBERT


In [42]:
!pip3 install -U numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 167.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4


In [1]:
import pickle
with open('../synthetic_data/aspect_query_fetched_positives.pkl','rb') as f:
    aspect_query_res=pickle.load(f)

In [2]:
aspect_query_res.keys()

dict_keys(['Industry', 'Customer Segment', 'Products & Services', 'Business Model', 'Technology Used', 'Revenue Model', 'Customer Industry'])

In [55]:
aspect_prompt={
'Industry':'Please ONLY qualify Companies that are providing solutions as a part of the given industry.Customer industry of a company is the industry to which company sells its solutions. DO NOT QUALIFY COMPANIES WHICH ALSO HAVE THE SAME CUSTOMER INDUSTRY SINCE WE WILL BE USING COMPANIES WITH SAME CUSTOMER INDUSTRY AS NEGATIVES FOR COMPANIES WITH SAME INDUSTRY',
'Customer Segment':'Customer Segment describes the class of customers that a company has',
'Products & Services':'',
'Business Model':'',
'Technology Used':'',
'Revenue Model':'',
'Customer Industry':'Please ONLY qualify Companies that are providing solutions to the provided industry as their customer industry.Customer industry of a company is the industry to which company sells its solutions. DO NOT QUALIFY COMPANIES WHICH ALSO HAVE THE SAME INDUSTRY SINCE WE WILL BE USING COMPANIES WITH SAME INDUSTRY AS NEGATIVES FOR COMPANIES WITH SAME CUSTOMER INDUSTRY'}

In [56]:
from dep.response_formats import FilteringResponse

In [57]:
# res=llm.get_text_completion(prompt,0)['choices'][0]['message']['content']
import json
def filter_companies(aspect,query,factsheet,response_format=FilteringResponse):
    
    success=False
    
    sys_msg_path1='../prompts/factsheet_relevance_check.txt'
    system_message1=open(sys_msg_path1,'r').read()
    llm1=OpenAIChatCompletion(model='gpt-4o',system_message=system_message1)
    
    prompt1=aspect_prompt[aspect] + f'Query: {aspect} - {query}\nFactsheet: {factsheet}'
    

    temperature=0
    
    res1=llm1.structured_response(prompt1,temperature,response_format)
    try:
        r1=json.loads(res1)
        success=True
        return success,r1
        
    except Exception as e:
        
        print(f'failed for {aspects} - {e}')
        success=False
        return success,{}

In [91]:
# Load the BERT tokenizer
model_name="bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
from transformers import BertTokenizer

def count_tokens(text, model_name="bert-base-uncased"):
    """
    Count the number of tokens in a text using BERT tokenizer.
    
    Args:
        text (str): The input text to tokenize and count
        model_name (str): The specific BERT model tokenizer to use
                         (default: 'bert-base-uncased')
    
    Returns:
        int: The number of tokens in the text
    """
    
    
    # Tokenize the text
    tokens = tokenizer.encode(text, add_special_tokens=True)
    
    # Return the count of tokens
    return len(tokens)

count_tokens(remove_text_before_pattern(factsheet))

Token indices sequence length is longer than the specified maximum sequence length for this model (559 > 512). Running this sequence through the model will result in indexing errors


559

In [100]:
import re

def remove_text_before_pattern(text):
    return re.sub(r'^.*?\* \*', '* *', text, count=1, flags=re.DOTALL).split('* * overall assessment :* *')[0]

In [102]:
from tqdm import tqdm 
for aspect,q_dict in tqdm(aspect_query_res.items()):
    for query,res in tqdm(q_dict.items()):
        res['token_count']=res.descriptions.apply(lambda x:count_tokens(remove_text_before_pattern(x)))
        aspect_query_res[aspect][query]=res

  0%|          | 0/7 [00:00<?, ?it/s]
%|          | 0/99 [00:00<?, ?it/s]
%|          | 1/99 [00:00<01:23,  1.17it/s]
%|▏         | 2/99 [00:01<01:22,  1.17it/s]
%|▎         | 3/99 [00:02<01:23,  1.15it/s]
%|▍         | 4/99 [00:03<01:20,  1.18it/s]
%|▌         | 5/99 [00:04<01:14,  1.26it/s]
%|▌         | 6/99 [00:05<01:18,  1.19it/s]
%|▋         | 7/99 [00:05<01:18,  1.17it/s]
%|▊         | 8/99 [00:06<01:19,  1.14it/s]
%|▉         | 9/99 [00:07<01:17,  1.16it/s]
%|█         | 10/99 [00:08<01:16,  1.17it/s]
%|█         | 11/99 [00:09<01:15,  1.17it/s]
%|█▏        | 12/99 [00:10<01:15,  1.16it/s]
%|█▎        | 13/99 [00:11<01:13,  1.17it/s]
%|█▍        | 14/99 [00:11<01:12,  1.17it/s]
%|█▌        | 15/99 [00:12<01:12,  1.16it/s]
%|█▌        | 16/99 [00:13<01:11,  1.16it/s]
%|█▋        | 17/99 [00:14<01:11,  1.15it/s]
%|█▊        | 18/99 [00:15<01:10,  1.14it/s]
%|█▉        | 19/99 [00:16<01:06,  1.20it/s]
%|██        | 20/99 [00:17<01:05,  1.20it/s]
%|██        | 21/99 [00:17<01:04,  

In [112]:
# print(remove_text_before_pattern(aspect_query_res[test_aspect][test_query][aspect_query_res[test_aspect][test_query].token_count>600].loc[9].descriptions))
len(aspect_query_res[test_aspect][test_query][aspect_query_res[test_aspect][test_query].token_count>650])

14

In [114]:
print(aspect_query_res[test_aspect][test_query][aspect_query_res[test_aspect][test_query].token_count>650].descriptions.iloc[0])

international trading companies, international presence, logistics sectors, international regulations, international trade, api connectors, financial services compliance, export transactions, business litigation, international legal professionals, international events, key authorities, business banking, logistics services, key competitors, business management functions, trade secret protection, business services, business printing, business loan, key differentiators, partner program, analytics services, business loans & lines of, business intelligence software, debt collection, business checking, business customers, import containers, business software solutions, debt recovery, lenders, business clients, facilitation services, logistics industry, business profile, logistics platform, business management support, business travel management solutions, business model, payment services, payment processing volume, logistics management, 
 okay , here 's the final the company company fact she

In [117]:
### filtering oput factsheets that exceed max token limit of colbert: keeping the threshold a bit relaxed since the information in the end is usually junk and not needed anyway
for aspect,q_dict in tqdm(aspect_query_res.items()):
    for query,res in tqdm(q_dict.items()):
        res=res[res.token_count<=650]
        aspect_query_res[aspect][query]=res
        print(len(res),aspect,query)

100%|██████████| 99/99 [00:00<00:00, 3113.62it/s]


74 Industry Entertainment
68 Industry Real Estate
64 Industry Biotechnology
80 Industry Insurance
94 Industry Legal
64 Industry Defense
78 Industry Telecommunications
65 Industry Utilities
77 Industry Pharmaceutical
77 Industry Hospitality
66 Industry Automotive
71 Industry Energy
77 Industry Technology
75 Industry Banking
65 Industry Fashion
74 Industry Chemicals
66 Industry Food and Beverage
65 Industry Renewable Energy
90 Industry Mining
80 Industry Logistics
82 Industry Beauty
83 Industry Agriculture
77 Industry Media
76 Industry Construction
72 Industry Consumer Electronics
68 Industry Education
86 Industry Finance
82 Industry Manufacturing
81 Industry Aerospace
73 Industry Healthcare
84 Industry Transportation
83 Industry Retail
80 Industry Public Sector
82 Industry Non-Profit
75 Industry Sports
75 Industry Gaming
78 Industry Paper and Pulp
77 Industry Fishing
78 Industry Furniture
82 Industry Marine
83 Industry Printing
84 Industry Advertising
77 Industry Textiles
79 Industry Ce


100%|██████████| 102/102 [00:00<00:00, 3183.11it/s]


87 Customer Segment Financial institutions
84 Customer Segment Pet care services
86 Customer Segment Beauty salons
77 Customer Segment Sports teams
84 Customer Segment Doctors
88 Customer Segment Pharmaceutical companies
81 Customer Segment Small businesses
78 Customer Segment Fitness centers
88 Customer Segment Government agencies
93 Customer Segment Logistics companies
79 Customer Segment Fashion brands
81 Customer Segment Automotive industry
86 Customer Segment Hospitality industry
91 Customer Segment Manufacturing companies
84 Customer Segment Travel agencies
83 Customer Segment Retailers
83 Customer Segment Consumer electronics
84 Customer Segment Lawyers
78 Customer Segment Healthcare providers
79 Customer Segment E-commerce businesses
84 Customer Segment Event planners
91 Customer Segment Energy companies
84 Customer Segment Restaurants
80 Customer Segment Media companies
79 Customer Segment Educational institutions
80 Customer Segment Telecommunications providers
81 Customer Se


 43%|████▎     | 3/7 [00:00<00:00, 29.00it/s]

...

73 Technology Used Augmented reality
63 Technology Used Blockchain
75 Technology Used AI
83 Technology Used Computer vision
80 Technology Used Natural language processing
73 Technology Used Chatbots
87 Technology Used Neural networks
59 Technology Used Biometrics
81 Technology Used Deep learning
75 Technology Used RPA (Robotic Process Automation)
72 Technology Used IoT
71 Technology Used Cybersecurity solutions
67 Technology Used Edge computing
81 Technology Used Voice recognition
64 Technology Used Smart contracts
77 Technology Used Quantum computing
83 Technology Used Facial recognition
75 Technology Used 5G technology
69 Technology Used Geolocation services
78 Technology Used Autonomous vehicles
81 Technology Used Machine learning
80 Technology Used Robotics
70 Technology Used Big data analytics
83 Technology Used Digital twins
64 Technology Used Cryptography
74 Technology Used Microservices architecture
63 Technology Used Wearable technology
71 Technology Used 3D printing
76 Techno


100%|██████████| 25/25 [00:00<00:00, 3167.14it/s]


80 Revenue Model Pay-per-use
89 Revenue Model Subscription-based
86 Revenue Model Marketplace
87 Revenue Model Commission
90 Revenue Model Licensing
83 Revenue Model Crowdsourcing
90 Revenue Model Freemium
79 Revenue Model Consulting fees
89 Revenue Model PaaS
88 Revenue Model SaaS
89 Revenue Model Royalties
80 Revenue Model Advertising
82 Revenue Model Wholesale
86 Revenue Model Retail
87 Revenue Model Affiliate
91 Revenue Model Direct sales
82 Revenue Model Franchise
80 Revenue Model Membership
91 Revenue Model B2C
81 Revenue Model B2B
72 Revenue Model Microtransactions
86 Revenue Model Sponsorship
86 Revenue Model Leasing
84 Revenue Model In-app purchases
83 Revenue Model Donation



100%|██████████| 7/7 [00:00<00:00, 41.98it/s]

84 Customer Industry Automotive
92 Customer Industry Mining
78 Customer Industry Retail
82 Customer Industry Banking
78 Customer Industry Food and Beverage
89 Customer Industry Finance
79 Customer Industry Entertainment
82 Customer Industry Fashion
89 Customer Industry Logistics
86 Customer Industry Agriculture
79 Customer Industry Transportation
80 Customer Industry Utilities
79 Customer Industry Insurance
84 Customer Industry Aerospace
84 Customer Industry Manufacturing
88 Customer Industry Telecommunications
79 Customer Industry Real Estate
83 Customer Industry Renewable Energy
94 Customer Industry Technology
91 Customer Industry Defense
95 Customer Industry Energy
88 Customer Industry Hospitality
93 Customer Industry Chemicals
88 Customer Industry Education
86 Customer Industry Healthcare
79 Customer Industry Construction
87 Customer Industry Consumer Electronics
83 Customer Industry Pharmaceutical
80 Customer Industry Beauty
90 Customer Industry Biotechnology
84 Customer Industry 

In [118]:
with open('../synthetic_data/aspect_query_fetched_positives_filtered_tokc.pkl','wb') as f:
    pickle.dump(aspect_query_res,f)

In [116]:
test_aspect='Customer Industry'
test_aspect='Business Model'
test_aspect='Technology Used'
test_aspect='Revenue Model'
# 'Mining', 'Retail', 'Banking'
test_query='Automotive'
test_query='Mining'
test_query='Banking'
test_query='B2B'
test_query='Computer vision'
test_query='Pay-per-use'
test_query='Marketplace'
aspect_query_res[test_aspect].keys()
factsheet=aspect_query_res[test_aspect][test_query].descriptions.iloc[10]

In [82]:
aspect_query_res['Revenue Model'].keys()

dict_keys(['Pay-per-use', 'Subscription-based', 'Marketplace', 'Commission', 'Licensing', 'Crowdsourcing', 'Freemium', 'Consulting fees', 'PaaS', 'SaaS', 'Royalties', 'Advertising', 'Wholesale', 'Retail', 'Affiliate', 'Direct sales', 'Franchise', 'Membership', 'B2C', 'B2B', 'Microtransactions', 'Sponsorship', 'Leasing', 'In-app purchases', 'Donation'])

In [83]:
filter_companies(test_aspect,test_query,factsheet)

(True, {'decision': 'YES'})

In [85]:
print(remove_text_before_pattern(factsheet))

* * africa sokoni fact sheet * * 

 * * 1 .   company overview :* * africa sokoni is a company ( inferred from the   . the company   domain ) aiming to empower micro , small , and medium - sized enterprises ( msmes ) across africa .    their mission is to provide comprehensive support , encompassing financing , e - commerce solutions , and various value - added services . 

 * * 2 .   products and services :* * 

 * * * fintech services :* *   access to working capital through tailored financing solutions . 
 * * * e - commerce platform :* * a multi - vendor marketplace with features like merchant central , ready market , sales insights , and a white label program . 
 * * * value - added services :* * business consultancy , training programs , custom solutions , legal and compliance assistance , technology integration , and market research and insights . 
 * * * supply chain management :* * a state - of - the - art logistics and warehousing system designed to streamline and automate su

In [139]:

query_positve_dict={}
for aspect,q_dict in tqdm(aspect_query_res.items()):
    for query,res in q_dict.items():
        query_positve_dict[(aspect,query)]=[]
        
        factsheets=res['descriptions'].to_list()
        args=[([aspect,query,factsheet],{}) for factsheet in factsheets]

        res=parallel_execute(
                        func= filter_companies,
                        argument_sets=args,
                        max_workers = 128,
                        show_progress= True,
                        desc = "judging factsheets retrieved"
                    )
        query_positve_dict[(aspect,query)]=list(zip([remove_text_before_pattern(factsheet) for factsheet in factsheets],res))
        

judging factsheets retrieved: 100%|██████████| 74/74 [00:04<00:00, 16.81it/s]

judging factsheets retrieved: 100%|██████████| 68/68 [00:01<00:00, 47.84it/s]

judging factsheets retrieved: 100%|██████████| 64/64 [00:01<00:00, 36.44it/s]

judging factsheets retrieved: 100%|██████████| 80/80 [00:03<00:00, 23.33it/s]

judging factsheets retrieved: 100%|██████████| 94/94 [00:05<00:00, 18.50it/s]

judging factsheets retrieved: 100%|██████████| 64/64 [00:04<00:00, 12.86it/s]

judging factsheets retrieved: 100%|██████████| 78/78 [00:06<00:00, 12.47it/s]

judging factsheets retrieved: 100%|██████████| 65/65 [00:01<00:00, 36.73it/s]

judging factsheets retrieved: 100%|██████████| 77/77 [00:01<00:00, 41.06it/s]

judging factsheets retrieved: 100%|██████████| 77/77 [00:06<00:00, 11.46it/s]

judging factsheets retrieved: 100%|██████████| 66/66 [00:03<00:00, 17.84it/s]

judging factsheets retrieved: 100%|██████████| 71/71 [00:05<00:00, 12.66it/s]

judging factsheets retrieved: 100%|██████████| 77/77

In [140]:
with open('../synthetic_data/aspect_query_positives_filtered_tokc_llm.pkl','wb') as f:
    pickle.dump(query_positve_dict,f)

In [290]:
### synthesizing positives for consumer industry queries which have less than threshold positives 

# res=llm.get_text_completion(prompt,0)['choices'][0]['message']['content']
additional_prompt={'Industry':'''Please ONLY provide synthetic facthseets for companies that are providing solutions as a part of the given industry.
Customer industry of a company is the industry to which company sells its solutions. 
DO NOT ADD FACTSHEET WHICH HAVE THE SAME CUSTOMER INDUSTRY SINCE WE WILL BE USING COMPANIES WITH SAME CUSTOMER INDUSTRY 
AS NEGATIVES FOR COMPANIES WITH SAME INDUSTRY''',
                   
                  'Customer Industry':'''Please ONLY provide synthetic facthseets for companies that are providing solutions to the given industry. i.e the customers of the company operate in the given industry
Customer industry of a company is the industry to which company sells its solutions. 
DO NOT ADD FACTSHEET WHICH HAVE THE SAME INDUSTRY AND CUSTOMER INDUSTRY SINCE WE WILL BE USING COMPANIES WITH SAME CUSTOMER INDUSTRY 
AS NEGATIVES FOR COMPANIES WITH SAME INDUSTRY'''}
from dep.response_formats import SyntheticPositiveFactsheetResponse

def generate_factsheets(query,n_fatchsheets,prompt1=None):
    success=False
    response_format=SyntheticPositiveFactsheetResponse
    sys_msg_path1='../prompts/synthetic_positive_factsheet.txt'
    system_message1=open(sys_msg_path1,'r').read()
    llm1=OpenAIChatCompletion(model='gpt-4o',system_message=system_message1)
    if prompt1 is None:
        prompt1=additional_prompt.get(f'{query[0]}','')+f"\nPlease provide {n_fatchsheets} facsheets for companies that are positives for the query: {query}"
    
    temperature=0.3
    
    res1=llm1.structured_response(prompt1,temperature,response_format)

    try:
        r1=json.loads(res1)['factsheets']

        success=True
        return success,r1
    except Exception as e:
        
        print(f'failed for {query} - {e}')
        success=False
        return success,[]

In [291]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import tqdm.notebook as tqdm
import json
def generate_factsheets_parallel(queries, n_factsheets=20, max_workers=10):
    """
    Run generate_factsheets in parallel for multiple queries.
    
    Args:
        queries (list): List of query strings to process
        n_factsheets (int): Number of factsheets to generate per query
        max_workers (int): Maximum number of parallel threads
        
    Returns:
        dict: Dictionary mapping queries to (success, factsheets) tuples
    """
    results = {}
    
    # Use tqdm for progress bar in Jupyter
    progress_bar = tqdm.tqdm(total=len(queries), desc="Processing queries")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Start the tasks
        future_to_query = {
            executor.submit(generate_factsheets, query, n_factsheets): query 
            for query in queries
        }
        
        # Process results as they complete
        for future in as_completed(future_to_query):
            query = future_to_query[future]
            try:
                success, factsheets = future.result()
                results[query] = (success, factsheets)
            except Exception as e:
                print(f"Query '{query}' generated an exception: {e}")
                results[query] = (False, [])
            
            progress_bar.update(1)
    
    progress_bar.close()
    return results

In [275]:
# list(query_positve_dict.values())[0]

In [158]:
rows=[]

for query_asp, factsheet_res in query_positve_dict.items():
    rows.extend([{'aspect':query_asp[0],'query':query_asp[1],'factsheet':factsheet_res[j][0],'judge_res':factsheet_res[j][1]} for j in range(len(factsheet_res))])



In [159]:


import pandas as pd
judge_res_df=pd.DataFrame(rows)

In [161]:
judge_res_df['judge_decision']=judge_res_df.judge_res.apply(lambda x:x[1]['decision'])

In [198]:
NO_df=judge_res_df[judge_res_df.judge_decision=='NO']
positives_df=judge_res_df[judge_res_df.judge_decision=='YES']

In [207]:
# NO_df

In [204]:
venture_df=NO_df[(NO_df.factsheet.str.lower().str.contains('venture')) | (NO_df.factsheet.str.lower().str.contains('capital'))]

In [221]:
factsheets=venture_df.factsheet.to_list()
### judging venture as true or False
args=[(['Industry','Venture Capital',factsheet],{}) for factsheet in factsheets]
bs=128
res=[]
for i in tqdm(range(0,len(args),bs)):
    
    _res=parallel_execute(
                    func= filter_companies,
                    argument_sets=args[i:i+bs],
                    max_workers = 128,
                    show_progress= True,
                    desc = "judging factsheets retrieved"
                )
    res.extend(_res)

100%|██████████| 47/47 [03:45<00:00,  4.81s/it]


In [222]:
venture_df['judge_res']=res
venture_df['judge_decision']=venture_df.judge_res.apply(lambda x:x[1]['decision'])

/tmp/ipykernel_31231/646181627.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  venture_df['judge_res']=res
/tmp/ipykernel_31231/646181627.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  venture_df['judge_decision']=venture_df.judge_res.apply(lambda x:x[1]['decision'])


In [240]:
venture_df['new_query']='Venture Capital'
venture_df['new_aspect']='Industry'

/tmp/ipykernel_31231/2404367603.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  venture_df['new_query']='Venture Capital'
/tmp/ipykernel_31231/2404367603.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  venture_df['new_aspect']='Industry'


In [318]:
venture_df.to_pickle('../synthetic_data/venture_df.pkl')

In [248]:
judge_res_df=pd.concat([judge_res_df,venture_df[['new_aspect', 'new_query', 'factsheet', 'judge_res', 'judge_decision']].rename(columns={'new_aspect':'aspect','new_query':'query'})])

In [251]:
judge_res_df


,aspect,query,factsheet,judge_res,judge_decision
0,Industry,Entertainment,* * 2112inc company fact sheet * * \n\n * * co...,"(True, {'decision': 'NO'})",NO
1,Industry,Entertainment,* * company fact sheet : oneragtime * * \n\n *...,"(True, {'decision': 'NO'})",NO
2,Industry,Entertainment,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'NO'})",NO
3,Industry,Entertainment,* * company fact sheet : the company ( the com...,"(True, {'decision': 'NO'})",NO
4,Industry,Entertainment,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'NO'})",NO
...,...,...,...,...,...
37069,Industry,Venture Capital,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'NO'})",NO
37099,Industry,Venture Capital,* * crisbro company fact sheet * * \n\n * * co...,"(True, {'decision': 'NO'})",NO
37114,Industry,Venture Capital,* * company fact sheet : asyx * * \n\n * * 1 ....,"(True, {'decision': 'NO'})",NO
37158,Industry,Venture Capital,"* * liqun commercial group co . , ltd . fact...","(True, {'decision': 'NO'})",NO


In [253]:
# true_counts

In [254]:
true_counts=judge_res_df[judge_res_df.judge_decision=='YES'].groupby(['aspect','query']).judge_decision.value_counts().reset_index()

In [261]:

true_counts[(true_counts['count']<30) & (true_counts['aspect'].isin(['Customer Industry','Industry']))]

,aspect,query,judge_decision,count
9,Customer Industry,Adtech,YES,22
11,Customer Industry,Aerospace,YES,4
12,Customer Industry,Agriculture,YES,25
13,Customer Industry,Agritech,YES,20
14,Customer Industry,Artificial Intelligence,YES,1
...,...,...,...,...
240,Industry,Satellite Technology,YES,1
244,Industry,Sports,YES,7
246,Industry,Telecommunications,YES,20
248,Industry,Tobacco,YES,3


In [267]:
regeneration_df=pd.DataFrame(columns=['aspect','query','n_docs'])

reg_less_positives=true_counts[(true_counts['count']<10) | ((true_counts['count']<20) & (true_counts.aspect.isin(['Customer Industry','Industry'])))]
reg_less_positives['n_docs']=reg_less_positives['count']
reg_less_positives.loc[reg_less_positives.aspect.isin(['Customer Industry','Industry']),'n_docs']=20-reg_less_positives.loc[reg_less_positives.aspect.isin(['Customer Industry','Industry']),'count']
reg_less_positives.loc[~reg_less_positives.aspect.isin(['Customer Industry','Industry']),'n_docs']=10-reg_less_positives.loc[~reg_less_positives.aspect.isin(['Customer Industry','Industry']),'count']

regeneration_df=pd.concat([regeneration_df,reg_less_positives[['aspect','query','n_docs']]])

/tmp/ipykernel_31231/1354226784.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reg_less_positives['n_docs']=reg_less_positives['count']


In [336]:
queries=list(zip(regeneration_df.aspect.to_list(),regeneration_df['query'].to_list()))
args=[([queries[i],regeneration_df.iloc[i].n_docs],{'prompt1':None}) for i in range(len(queries))]

In [337]:

generated_factsheets=parallel_execute(
                        func= generate_factsheets,
                        argument_sets=args,
                        max_workers = 256,
                        show_progress= True,
                        desc = "generating factsheets"
                    )

[([('Business Model', 'Direct sales'), 3], {'prompt1': None}),
 ([('Business Model', 'Direct sales'), 3], {'prompt1': None}),
 ([('Business Model', 'Direct sales'), 3], {'prompt1': None}),
 ([('Business Model', 'Franchise'), 6], {'prompt1': None}),
 ([('Business Model', 'Franchise'), 6], {'prompt1': None})]

In [340]:

aspects=['Revenue model','Customer Segment','Customer Segment','Industry','Customer Industry','Industry','Customer Industry']
quries=['Usage based','K-12','Startups','Cybersecurity','Cybersecurity','Quantum Computing','Quantum Computing']
new_additions=pd.DataFrame({'aspect':aspects,'query':quries,'n_docs':[10]*(len(aspects)-3)+[30,30,30]})

In [341]:
queries=list(zip(new_additions.aspect.to_list(),new_additions['query'].to_list()))
args=[([queries[i],new_additions.iloc[i].n_docs],{'prompt1':None}) for i in range(len(queries))]

new_additions_generated_factsheets=parallel_execute(
                        func= generate_factsheets,
                        argument_sets=args,
                        max_workers = 7,
                        show_progress= True,
                        desc = "generating factsheets"
                    )


generating factsheets: 100%|██████████| 7/7 [01:47<00:00, 15.41s/it]


In [378]:
quntum_computing_facsheet_gen=generate_factsheets(('Customer Industry','Quantum Computing'),30)

In [343]:
new_additions['factsheet']=new_additions_generated_factsheets

In [344]:
new_additions['factsheet']=new_additions['factsheet'].apply(lambda x:x[1])

In [345]:
new_additions=new_additions.explode('factsheet')

In [348]:
new_additions

,aspect,query,n_docs,factsheet,judge_decision
0,Revenue model,Usage based,10,**Company Name:** CloudScale Technologies \n*...,YES
0,Revenue model,Usage based,10,**Company Name:** DataStream Analytics \n**We...,YES
0,Revenue model,Usage based,10,**Company Name:** GreenEnergy Solutions \n**W...,YES
0,Revenue model,Usage based,10,**Company Name:** FlexiNet Communications \n*...,YES
0,Revenue model,Usage based,10,**Company Name:** AquaPure Technologies \n**W...,YES
...,...,...,...,...,...
6,Custom Industry,Quantum Computing,30,**Company Name:** QuantumRetail Solutions \n*...,YES
6,Custom Industry,Quantum Computing,30,**Company Name:** QuantumTransport Labs \n**W...,YES
6,Custom Industry,Quantum Computing,30,**Company Name:** QuantumMedia Solutions \n**...,YES
6,Custom Industry,Quantum Computing,30,**Company Name:** QuantumSecurity Labs \n**We...,YES


In [347]:
new_additions['judge_decision']='YES'

In [300]:
regeneration_df['factsheet']=generated_factsheets

In [302]:
regeneration_df['factsheet']=regeneration_df['factsheet'].apply(lambda x:x[1])

In [304]:
regeneration_df=regeneration_df.explode('factsheet')

In [350]:
regeneration_df=pd.concat([regeneration_df,new_additions])

In [379]:
regeneration_df=regeneration_df[regeneration_df.aspect!='Custom Industry']

In [380]:
regeneration_df

,aspect,query,n_docs,factsheet,judge_decision
2,Business Model,Direct sales,3,**Company Name:** DirectTech Solutions \n**We...,YES
2,Business Model,Direct sales,3,**Company Name:** GreenHome Innovations \n**W...,YES
2,Business Model,Direct sales,3,**Company Name:** HealthDirect Supplies \n**W...,YES
3,Business Model,Franchise,6,**Company Name:** GreenLeaf Cafe \n**Website:...,YES
3,Business Model,Franchise,6,**Company Name:** FitZone Gyms \n**Website:**...,YES
...,...,...,...,...,...
5,Industry,Quantum Computing,30,**Company Name:** QuantumFlow Systems \n**Web...,YES
5,Industry,Quantum Computing,30,**Company Name:** QuantumCircuit Technologies ...,YES
5,Industry,Quantum Computing,30,**Company Name:** QuantumCompute Solutions \n...,YES
5,Industry,Quantum Computing,30,**Company Name:** QuantumFusion Technologies ...,YES


In [385]:
quntum_computing_facsheet_gen_df=pd.DataFrame({'factsheet':quntum_computing_facsheet_gen[1],
                                               'aspect':['Customer Industry']*len(quntum_computing_facsheet_gen[1]),
                                                'query':['Quantum Computing']*len(quntum_computing_facsheet_gen[1]),
                                              'judge_decision':['YES']*len(quntum_computing_facsheet_gen[1])})

In [388]:
regeneration_df.to_pickle('../synthetic_data/generated_factsheets.pkl')

In [389]:
regeneration_df

,aspect,query,n_docs,factsheet,judge_decision
2,Business Model,Direct sales,3,**Company Name:** DirectTech Solutions \n**We...,YES
2,Business Model,Direct sales,3,**Company Name:** GreenHome Innovations \n**W...,YES
2,Business Model,Direct sales,3,**Company Name:** HealthDirect Supplies \n**W...,YES
3,Business Model,Franchise,6,**Company Name:** GreenLeaf Cafe \n**Website:...,YES
3,Business Model,Franchise,6,**Company Name:** FitZone Gyms \n**Website:**...,YES
...,...,...,...,...,...
23,Customer Industry,Quantum Computing,NaN,**Company Name:** QuantumDev Integration \n**...,YES
24,Customer Industry,Quantum Computing,NaN,**Company Name:** QuantumSecure Integration \...,YES
25,Customer Industry,Quantum Computing,NaN,**Company Name:** QuantumAI Solutions \n**Web...,YES
26,Customer Industry,Quantum Computing,NaN,**Company Name:** QuantumCompute Tools \n**We...,YES


In [312]:
# regeneration_df['judge_decision']='YES'

In [353]:
judge_res_df=pd.concat([judge_res_df,regeneration_df[['aspect', 'query', 'factsheet', 'judge_decision']]])

In [392]:
judge_res_df.to_pickle('../synthetic_data/final_factsheet_judged_df.pkl')

In [393]:
judge_res_df

,aspect,query,factsheet,judge_res,judge_decision
0,Industry,Entertainment,* * 2112inc company fact sheet * * \n\n * * co...,"(True, {'decision': 'NO'})",NO
1,Industry,Entertainment,* * company fact sheet : oneragtime * * \n\n *...,"(True, {'decision': 'NO'})",NO
2,Industry,Entertainment,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'NO'})",NO
3,Industry,Entertainment,* * company fact sheet : the company ( the com...,"(True, {'decision': 'NO'})",NO
4,Industry,Entertainment,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'NO'})",NO
...,...,...,...,...,...
23,Customer Industry,Quantum Computing,**Company Name:** QuantumDev Integration \n**...,NaN,YES
24,Customer Industry,Quantum Computing,**Company Name:** QuantumSecure Integration \...,NaN,YES
25,Customer Industry,Quantum Computing,**Company Name:** QuantumAI Solutions \n**Web...,NaN,YES
26,Customer Industry,Quantum Computing,**Company Name:** QuantumCompute Tools \n**We...,NaN,YES


In [394]:
positives_df=judge_res_df[judge_res_df.judge_decision=='YES']

In [395]:
positives_df.to_pickle('../synthetic_data/positives.pkl')

In [396]:
positives_df.groupby(['aspect','query']).judge_decision.value_counts()

aspect           query                judge_decision
Business Model   B2B                  YES               67
                 B2C                  YES               25
                 Direct sales         YES               10
                 Franchise            YES               10
                 Freemium             YES               10
                                                        ..
Technology Used  Robotics             YES               78
                 Smart contracts      YES               24
                 Virtual reality      YES               65
                 Voice recognition    YES               47
                 Wearable technology  YES               31
Name: count, Length: 410, dtype: int64

In [397]:
positives_df

,aspect,query,factsheet,judge_res,judge_decision
16,Industry,Entertainment,* * company name :* * potato productions \n\n ...,"(True, {'decision': 'YES'})",YES
28,Industry,Entertainment,* * company name :* * the company group \n\n *...,"(True, {'decision': 'YES'})",YES
54,Industry,Entertainment,* * company fact sheet : the company managemen...,"(True, {'decision': 'YES'})",YES
55,Industry,Entertainment,* * the company fact sheet * * \n\n * * compan...,"(True, {'decision': 'YES'})",YES
106,Industry,Real Estate,* * company fact sheet : styrkonstruktion * * ...,"(True, {'decision': 'YES'})",YES
...,...,...,...,...,...
23,Customer Industry,Quantum Computing,**Company Name:** QuantumDev Integration \n**...,NaN,YES
24,Customer Industry,Quantum Computing,**Company Name:** QuantumSecure Integration \...,NaN,YES
25,Customer Industry,Quantum Computing,**Company Name:** QuantumAI Solutions \n**Web...,NaN,YES
26,Customer Industry,Quantum Computing,**Company Name:** QuantumCompute Tools \n**We...,NaN,YES


In [399]:
print(positives_df[(positives_df.aspect=='Customer Industry') & (positives_df.judge_res.isna())].iloc[0].factsheet)

**Company Name:** AeroTech Solutions  
**Website:** aerotechsolutions.com  
**Industry:** Develops advanced materials and composites for high-performance applications.  
**Target Audience:** Aerospace manufacturers, including aircraft and spacecraft producers seeking innovative material solutions.  
**Products & Services:**  
- **NanoComposite Materials:** Lightweight, durable materials designed to enhance fuel efficiency and performance in aerospace applications.  
- **Thermal Protection Systems:** Advanced coatings and materials for heat resistance in high-speed aerospace vehicles.  
**Unique Selling Propositions (USPs) & Key Differentiators:**  
- Proprietary technology that reduces weight while maintaining structural integrity.  
- Customizable solutions tailored to specific aerospace requirements.  
**Business Model:** B2B with project-based contracts and long-term supply agreements.  
**Technology Used:** Nanotechnology, advanced polymer science, and thermal engineering.  
**Reve

In [370]:
print(positives_df.iloc[-10].factsheet)

**Company Name:** QuantumSecurity Solutions  
**Website:** quantumsecurity.com  
**Industry:** Specializes in quantum computing applications for cybersecurity and data protection.  
**Target Audience:** Financial institutions, government agencies, and enterprises seeking advanced tools for data security and threat detection.  
**Products & Services:**  
- **QuantumThreat Detector:** A tool that uses quantum algorithms to detect and prevent cyber threats with unprecedented speed and accuracy.  
- **Data Quantum Protector:** Software for securing sensitive data and communications with quantum encryption.  
**Unique Selling Propositions (USPs) & Key Differentiators:**  
- Unmatched speed and accuracy in threat detection and prevention.  
- Integration with existing security systems for seamless operation.  
**Business Model:** B2B with software licensing and consulting services.  
**Technology Used:** Quantum computing, cybersecurity frameworks, and data protection systems.  
**Revenue Mo

In [400]:
negatives_df=pd.DataFrame(columns=['aspect','query','factsheet'])

In [402]:
### venture Df was only made of rows that were not correct documents for aspect, query
## what we will be doing now is that we will be setting the correct venture capital docs as negatives for individual 
positive_venture_df=venture_df[venture_df.judge_decision=='YES']

In [410]:
(positive_venture_df.groupby(['aspect','query']).judge_decision.value_counts().sort_values().reset_index()['count']>10).sum()

np.int64(68)

In [411]:
positive_venture_df

,aspect,query,factsheet,judge_res,judge_decision,new_query,new_aspect
1,Industry,Entertainment,* * company fact sheet : oneragtime * * \n\n *...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry
2,Industry,Entertainment,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry
11,Industry,Entertainment,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry
14,Industry,Entertainment,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry
17,Industry,Entertainment,* * company name :* * the company inc . \n\n *...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry
...,...,...,...,...,...,...,...
37018,Customer Industry,Retailtech,* * company fact sheet : shape capital * * \n\...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry
37019,Customer Industry,Retailtech,* * the company fact sheet * * \n\n * * 1 . ...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry
37020,Customer Industry,Retailtech,* * company name :* * aerox ventures \n\n * * ...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry
37021,Customer Industry,Retailtech,* * company name :* * golden section \n * * we...,"(True, {'decision': 'YES'})",YES,Venture Capital,Industry


In [1377]:
negatives_df=positive_venture_df[['aspect','query','factsheet']]

In [1378]:
negatives_df['strategy']='venture'

/tmp/ipykernel_31231/2478520030.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  negatives_df['strategy']='venture'


In [1379]:
customer_negatives=positives_df[positives_df.aspect.str.contains('Industry')][['aspect','query','factsheet']]

In [1380]:
customer_negatives['aspect']=customer_negatives['aspect'].apply(lambda x:'Customer Industry' if x=='Industry' else 'Industry')

In [1381]:
customer_negatives

,aspect,query,factsheet
16,Customer Industry,Entertainment,* * company name :* * potato productions \n\n ...
28,Customer Industry,Entertainment,* * company name :* * the company group \n\n *...
54,Customer Industry,Entertainment,* * company fact sheet : the company managemen...
55,Customer Industry,Entertainment,* * the company fact sheet * * \n\n * * compan...
106,Customer Industry,Real Estate,* * company fact sheet : styrkonstruktion * * ...
...,...,...,...
23,Industry,Quantum Computing,**Company Name:** QuantumDev Integration \n**...
24,Industry,Quantum Computing,**Company Name:** QuantumSecure Integration \...
25,Industry,Quantum Computing,**Company Name:** QuantumAI Solutions \n**Web...
26,Industry,Quantum Computing,**Company Name:** QuantumCompute Tools \n**We...


In [871]:
# positives_df[positives_df['query']=='Venture Capital']

In [1362]:
### curating negatives for Technology Used,  Revenue Model, Business Model

# from dep.response_formats import HardNegativeFactsheetResponse
aspect_system_message_file={'Technology Used':'../prompts/tech_used_hard_negatives.txt',
                           'Revenue Model':'../prompts/revenue_model_hard_negative_prompt.txt',
                            'Business Model':'../prompts/business_model_hard_negatives_prompt.txt',
                            'Customer Segment': '../prompts/customer_segment.txt',
                            'Customer Industry':'../prompts/customer_industry.txt',
                            'Industry':'../prompts/Industry.txt',
                            'Products & Services': '../prompts/Products&Services.txt'
                           }

def generate_factsheets(query,n_fatchsheets,aspect):
    success=False
    response_format=HardNegativeFactsheetResponse
    
    sys_msg_path1=aspect_system_message_file[aspect]
    
    system_message1=open(sys_msg_path1,'r').read()
    # print(system_message1)|
    llm1=OpenAIChatCompletion(model='gpt-4o',system_message=system_message1)
    

    prompt1=f"\nPlease provide {n_fatchsheets} facsheets for companies that are hard negatives for the query: {query}. Keep each Factsheet less than 200-250 words, Since the model to be trained has a token limit."
    
    temperature=0.3
    
    res1=llm1.structured_response(prompt1,temperature,response_format)

    try:
        r1=json.loads(res1)['factsheets']

        success=True
        return success,r1
    except Exception as e:
        
        print(f'failed for {query} - {e}')
        success=False
        return success,[]

In [1363]:
positives_df[positives_df.aspect.str.contains('Tech')]['query'].unique()

array(['Augmented reality', 'Blockchain', 'AI', 'Computer vision',
       'Natural language processing', 'Chatbots', 'Neural networks',
       'Biometrics', 'Deep learning', 'RPA (Robotic Process Automation)',
       'IoT', 'Cybersecurity solutions', 'Edge computing',
       'Voice recognition', 'Smart contracts', 'Quantum computing',
       'Facial recognition', '5G technology', 'Geolocation services',
       'Autonomous vehicles', 'Machine learning', 'Robotics',
       'Big data analytics', 'Digital twins', 'Cryptography',
       'Microservices architecture', 'Wearable technology', '3D printing',
       'Virtual reality', 'Cloud computing'], dtype=object)

In [1364]:
factsheets=generate_factsheets(('Customer Segment','Legal Departments'),10,'Customer Segment')
# factsheets=generate_factsheets(('Products & Services','Augmented reality applications'),10,'Products & Services')
# factsheets=generate_factsheets(('Technology Used','3D printing'),10,'Technology Used')

In [1365]:
print(factsheets[1][0])

Company Name: FinTech Solutions  Website: fintechsolutions.com  Industry: Financial Technology  Target Audience: Small to medium-sized financial institutions  Products & Services:  - FinTech Suite: Comprehensive financial management tools  - FinTech Analytics: Data-driven insights for financial decision-making  Unique Selling Propositions (USPs) & Key Differentiators:  - Customizable financial solutions tailored to client needs  - Advanced security features for financial data protection  Business Model: B2B SaaS  Technology Used: Blockchain, AI-driven analytics  Revenue Model: Subscription-based  Competitive Landscape: Competes with Plaid, Stripe, and Square


In [1366]:
print(factsheets[1][5])

Company Name: SmartCity Solutions  Website: smartcitysolutions.com  Industry: Urban Development Technology  Target Audience: Municipal governments and urban planners  Products & Services:  - SmartCity Infrastructure: IoT-enabled urban management systems  - SmartCity Analytics: Data-driven urban planning tools  Unique Selling Propositions (USPs) & Key Differentiators:  - Comprehensive smart city solutions for urban efficiency  - Real-time data analytics for informed decision-making  Business Model: B2G (Business to Government)  Technology Used: IoT, Big Data analytics  Revenue Model: Government contracts and service fees  Competitive Landscape: Competes with Siemens, IBM Smart Cities, and Cisco


In [1367]:
print(factsheets[1][9])

Company Name: MediaTech Innovations  Website: mediatechinnovations.com  Industry: Media and Entertainment Technology  Target Audience: Broadcasting companies and media networks  Products & Services:  - MediaTech Streaming: High-quality streaming solutions  - MediaTech Analytics: Audience engagement and analytics tools  Unique Selling Propositions (USPs) & Key Differentiators:  - Scalable streaming technology for large audiences  - In-depth audience insights for content optimization  Business Model: B2B SaaS  Technology Used: Cloud-based streaming, AI-driven analytics  Revenue Model: Subscription and licensing fees  Competitive Landscape: Competes with Brightcove, Vimeo, and Kaltura


In [1368]:
# factsheets[1]

In [525]:
positives_df['consolidated_query']=positives_df.apply(lambda x:(x.aspect,x['query']),axis=1)

/tmp/ipykernel_31231/2929412718.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positives_df['consolidated_query']=positives_df.apply(lambda x:(x.aspect,x['query']),axis=1)


In [536]:
# positives_df.consolidated_query.unique()

In [ ]:
# query_negatives_dict={}

In [1338]:
for aspect in tqdm(['Customer Segment']):
    queries=list(positives_df[positives_df.aspect==aspect].consolidated_query.unique())

    args=[([queries[i],60,aspect],{}) for i in range(len(queries))]
    bs=20
    _query_negatives_dict={}
    
    for i in tqdm(range(0,60,20)):
        # basically calculating 3 times to get 60 in total
        
        
        args=[([queries[i],20,aspect],{}) for i in range(len(queries))]
            
        factsheets=parallel_execute(
                            func= generate_factsheets,
                            argument_sets=args,
                            max_workers = 32,
                            show_progress= True,
                            desc = "generating negative factsheets"
                        )

        for i in range(len(queries)):
            
            query=queries[i]
            factsheet=factsheets[i]
            
            if query in _query_negatives_dict:
                _query_negatives_dict[query]=_query_negatives_dict[query]+factsheet[1]
            else:
                _query_negatives_dict[query]=factsheet[1]

    query_negatives_dict={**query_negatives_dict,**_query_negatives_dict}


  0%|          | 0/1 [00:00<?, ?it/s]
%|          | 0/3 [00:00<?, ?it/s]

ting negative factsheets:   0%|          | 0/84 [00:00<?, ?it/s]

ting negative factsheets:   1%|          | 1/84 [00:18<25:03, 18.11s/it]

ting negative factsheets:   2%|▏         | 2/84 [00:18<10:21,  7.58s/it]

ting negative factsheets:   4%|▎         | 3/84 [00:18<05:54,  4.38s/it]

ting negative factsheets:   5%|▍         | 4/84 [00:19<03:59,  2.99s/it]

ting negative factsheets:   6%|▌         | 5/84 [00:19<02:34,  1.96s/it]

ting negative factsheets:   7%|▋         | 6/84 [00:20<01:59,  1.53s/it]

ting negative factsheets:   8%|▊         | 7/84 [00:20<01:27,  1.13s/it]

ting negative factsheets:  11%|█         | 9/84 [00:21<00:46,  1.62it/s]

ting negative factsheets:  12%|█▏        | 10/84 [00:21<00:36,  2.00it/s]

ting negative factsheets:  13%|█▎        | 11/84 [00:21<00:29,  2.51it/s]

ting negative factsheets:  15%|█▌        | 13/84 [00:21<00:23,  3.05it/s]

ting negative factsheets:  17%|█▋        | 

In [542]:
query_negatives_dict={}
for aspect in tqdm(['Technology Used','Revenue Model','Business Model','Customer Segment']):
    queries=list(positives_df[positives_df.aspect==aspect].consolidated_query.unique())
    
    args=[([queries[i],60,aspect],{}) for i in range(len(queries))]
    
    factsheets=parallel_execute(
                        func= generate_factsheets,
                        argument_sets=args,
                        max_workers = 32,
                        show_progress= True,
                        desc = "generating negative factsheets"
                    )
    
    
    _query_negatives_dict=dict(zip(queries,factsheets))
    
    query_negatives_dict={**query_negatives_dict,**_query_negatives_dict}
    

100%|██████████| 3/3 [08:35<00:00, 171.99s/it]


In [546]:
query_negatives_dict[('Technology Used', 'Augmented reality')]

(True,
 ['**Company Name:** CrystalTech Innovations  **Website:** crystaltech.com  **Industry:** Optical Solutions  **Target Audience:** Manufacturers in need of advanced optical components  **Products & Services:**  - Crystal Lenses: Precision lenses for industrial applications  - Optical Coatings: Anti-reflective coatings for various surfaces  **Unique Selling Propositions (USPs) & Key Differentiators:**  - High precision manufacturing processes  - Customizable optical solutions  **Business Model:** B2B  **Technology Used:** Advanced optics and laser technology, but no implementation of augmented reality solutions  **Revenue Model:** Project-based contracts  **Competitive Landscape:** Competes with other optical component manufacturers  **Overall Assessment:** While CrystalTech is at the forefront of optical solutions, their product line does not integrate augmented reality technology. They focus on traditional optics, which are unrelated to AR applications.',
  '**Company Name:** Vi

In [1369]:
with open('../synthetic_data/synthetic_negatives.pkl','wb') as f:
    pickle.dump(query_negatives_dict,f)

In [ ]:
for aspect in tqdm(['Customer Segment']):
    queries=list(positives_df[positives_df.aspect==aspect].consolidated_query.unique())
    
    args=[([queries[i],60,aspect],{}) for i in range(len(queries))]
    
    factsheets=parallel_execute(
                        func= generate_factsheets,
                        argument_sets=args,
                        max_workers = 50,
                        show_progress= True,
                        desc = "generating negative factsheets"
                    )
    
    
    _query_negatives_dict=dict(zip(queries,factsheets))
    
    query_negatives_dict={**query_negatives_dict,**_query_negatives_dict}

100%|██████████| 1/1 [05:00<00:00, 300.18s/it]


In [ ]:
$267.71 -> $260.22


In [581]:
with open('../synthetic_data/synthetic_negatives.pkl','wb') as f:
    pickle.dump(query_negatives_dict,f)

In [ ]:
254.47

In [675]:
for aspect in tqdm(['Customer Industry','Industry']):
    queries=list(positives_df[positives_df.aspect==aspect].consolidated_query.unique())
    
    args=[([queries[i],50,aspect],{}) for i in range(len(queries))]
    
    factsheets=parallel_execute(
                        func= generate_factsheets,
                        argument_sets=args,
                        max_workers = 50,
                        show_progress= True,
                        desc = "generating negative factsheets"
                    )
    
    
    _query_negatives_dict=dict(zip(queries,factsheets))
    
    query_negatives_dict={**query_negatives_dict,**_query_negatives_dict}

100%|██████████| 2/2 [05:39<00:00, 169.96s/it]


In [679]:
for aspect in tqdm(['Products & Services']):
    queries=list(positives_df[positives_df.aspect==aspect].consolidated_query.unique())
    
    args=[([queries[i],25,aspect],{}) for i in range(len(queries))]

    factsheets=parallel_execute(
                        func= generate_factsheets,
                        argument_sets=args,
                        max_workers = 25,
                        show_progress= True,
                        desc = "generating negative factsheets"
                    )
    
    
    _query_negatives_dict=dict(zip(queries,factsheets))
    
    query_negatives_dict={**query_negatives_dict,**_query_negatives_dict}

100%|██████████| 1/1 [02:56<00:00, 176.93s/it]


In [680]:
for aspect in tqdm(['Products & Services']):
    queries=list(positives_df[positives_df.aspect==aspect].consolidated_query.unique())
    
    args=[([queries[i],25,aspect],{}) for i in range(len(queries))]

    factsheets=parallel_execute(
                        func= generate_factsheets,
                        argument_sets=args,
                        max_workers = 25,
                        show_progress= True,
                        desc = "generating negative factsheets"
                    )
    
    
    _query_negatives_dict=dict(zip(queries,factsheets))
    
    for query, factsheets in _query_negatives_dict.items():
        if query in query_negatives_dict:
            print('appending')
            query_negatives_dict[query]=query_negatives_dict[query]+factsheets
        else:
            
            query_negatives_dict[query]=factsheets

100%|██████████| 1/1 [03:05<00:00, 185.18s/it]

appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending
appending


In [1216]:
query_negatives_dict.keys()

dict_keys([('Technology Used', 'Augmented reality'), ('Technology Used', 'Blockchain'), ('Technology Used', 'AI'), ('Technology Used', 'Computer vision'), ('Technology Used', 'Natural language processing'), ('Technology Used', 'Chatbots'), ('Technology Used', 'Neural networks'), ('Technology Used', 'Biometrics'), ('Technology Used', 'Deep learning'), ('Technology Used', 'RPA (Robotic Process Automation)'), ('Technology Used', 'IoT'), ('Technology Used', 'Cybersecurity solutions'), ('Technology Used', 'Edge computing'), ('Technology Used', 'Voice recognition'), ('Technology Used', 'Smart contracts'), ('Technology Used', 'Quantum computing'), ('Technology Used', 'Facial recognition'), ('Technology Used', '5G technology'), ('Technology Used', 'Geolocation services'), ('Technology Used', 'Autonomous vehicles'), ('Technology Used', 'Machine learning'), ('Technology Used', 'Robotics'), ('Technology Used', 'Big data analytics'), ('Technology Used', 'Digital twins'), ('Technology Used', 'Crypt

In [1228]:
positives_df[positives_df['query']=='Advertising'].aspect.unique()

array(['Industry', 'Revenue Model', 'Customer Industry'], dtype=object)

In [1231]:
len(query_negatives_dict[('Customer Industry', 'Advertising')][1])

48

In [1007]:
query_negatives_dict.keys()

dict_keys([('Technology Used', 'Augmented reality'), ('Technology Used', 'Blockchain'), ('Technology Used', 'AI'), ('Technology Used', 'Computer vision'), ('Technology Used', 'Natural language processing'), ('Technology Used', 'Chatbots'), ('Technology Used', 'Neural networks'), ('Technology Used', 'Biometrics'), ('Technology Used', 'Deep learning'), ('Technology Used', 'RPA (Robotic Process Automation)'), ('Technology Used', 'IoT'), ('Technology Used', 'Cybersecurity solutions'), ('Technology Used', 'Edge computing'), ('Technology Used', 'Voice recognition'), ('Technology Used', 'Smart contracts'), ('Technology Used', 'Quantum computing'), ('Technology Used', 'Facial recognition'), ('Technology Used', '5G technology'), ('Technology Used', 'Geolocation services'), ('Technology Used', 'Autonomous vehicles'), ('Technology Used', 'Machine learning'), ('Technology Used', 'Robotics'), ('Technology Used', 'Big data analytics'), ('Technology Used', 'Digital twins'), ('Technology Used', 'Crypt

In [681]:
with open('../synthetic_data/synthetic_negatives.pkl','wb') as f:
    pickle.dump(query_negatives_dict,f)

In [1382]:
set([key[0] for key in query_negatives_dict.keys()])

{'Business Model',
 'Customer Industry',
 'Customer Segment',
 'Industry',
 'Products & Services',
 'Revenue Model',
 'Technology Used'}

In [1383]:
# _query_negatives_dict.keys()

In [1384]:
rows=[]
for aspect_query, synthetic_negatives in query_negatives_dict.items():
    rows.extend([{'aspect':aspect_query[0],'query':aspect_query[1],'factsheet':factsheet} for factsheet in synthetic_negatives[1]])

In [1385]:
synthetic_negatives_df=pd.DataFrame(rows)

In [1386]:
customer_negatives['strategy']='rule'

In [1387]:
synthetic_negatives_df['strategy']='synthetic'

In [1388]:
synthetic_negatives_df

,aspect,query,factsheet,strategy
0,Technology Used,Augmented reality,**Company Name:** DataViz Analytics \n**Websi...,synthetic
1,Technology Used,Augmented reality,**Company Name:** AIToolbox \n**Website:** ai...,synthetic
2,Technology Used,Augmented reality,**Company Name:** CloudSync Solutions \n**Web...,synthetic
3,Technology Used,Augmented reality,**Company Name:** EcoTech Innovations \n**Web...,synthetic
4,Technology Used,Augmented reality,**Company Name:** FinTech Dynamics \n**Websit...,synthetic
...,...,...,...,...
18390,Products & Services,Online interior design services,**Company Name:** TechGadgets **Website:** te...,synthetic
18391,Products & Services,Online interior design services,**Company Name:** StyleSavvy **Website:** sty...,synthetic
18392,Products & Services,Online interior design services,**Company Name:** TravelTales **Website:** tr...,synthetic
18393,Products & Services,Online interior design services,**Company Name:** GreenEnergy **Website:** gr...,synthetic


In [1389]:
negatives_df=pd.concat([synthetic_negatives_df,customer_negatives,negatives_df])

In [1390]:
negatives_df

,aspect,query,factsheet,strategy
0,Technology Used,Augmented reality,**Company Name:** DataViz Analytics \n**Websi...,synthetic
1,Technology Used,Augmented reality,**Company Name:** AIToolbox \n**Website:** ai...,synthetic
2,Technology Used,Augmented reality,**Company Name:** CloudSync Solutions \n**Web...,synthetic
3,Technology Used,Augmented reality,**Company Name:** EcoTech Innovations \n**Web...,synthetic
4,Technology Used,Augmented reality,**Company Name:** FinTech Dynamics \n**Websit...,synthetic
...,...,...,...,...
37018,Customer Industry,Retailtech,* * company fact sheet : shape capital * * \n\...,venture
37019,Customer Industry,Retailtech,* * the company fact sheet * * \n\n * * 1 . ...,venture
37020,Customer Industry,Retailtech,* * company name :* * aerox ventures \n\n * * ...,venture
37021,Customer Industry,Retailtech,* * company name :* * golden section \n * * we...,venture


In [1391]:
# negatives_df.aspect.unique()

In [1392]:
# synthetic_negatives_df

In [1393]:
print(negatives_df[negatives_df.strategy=='rule'].iloc[-10].factsheet)

**Company Name:** QuantumDev Systems  
**Website:** quantumdevsystems.com  
**Industry:** Develops development tools and environments for quantum computing applications.  
**Target Audience:** Quantum computing developers and researchers needing robust development environments.  
**Products & Services:**  
- **Quantum IDE:** Integrated development environment tailored for quantum programming.  
- **Qubit Debugging Tools:** Tools for debugging and testing quantum algorithms.  
**Unique Selling Propositions (USPs) & Key Differentiators:**  
- Comprehensive development environment for quantum computing.  
- Advanced debugging tools for quantum algorithms.  
**Business Model:** B2B with software licensing and support services.  
**Technology Used:** Quantum programming and development technologies.  
**Revenue Model:** Licensing fees and support contracts.  
**Competitive Landscape:** Competes with general development tool providers but focuses on quantum-specific applications.  
**Overall

In [1394]:
negatives_df_group=negatives_df.groupby(['aspect','query','strategy'])

In [1395]:
negatives_df_group.get_group(('Technology Used', 'Augmented reality', 'synthetic'))

,aspect,query,factsheet,strategy
0,Technology Used,Augmented reality,**Company Name:** DataViz Analytics \n**Websi...,synthetic
1,Technology Used,Augmented reality,**Company Name:** AIToolbox \n**Website:** ai...,synthetic
2,Technology Used,Augmented reality,**Company Name:** CloudSync Solutions \n**Web...,synthetic
3,Technology Used,Augmented reality,**Company Name:** EcoTech Innovations \n**Web...,synthetic
4,Technology Used,Augmented reality,**Company Name:** FinTech Dynamics \n**Websit...,synthetic
5,Technology Used,Augmented reality,**Company Name:** HealthSync Technologies \n*...,synthetic
6,Technology Used,Augmented reality,**Company Name:** LogiTech Systems \n**Websit...,synthetic
7,Technology Used,Augmented reality,**Company Name:** EduTech Innovators \n**Webs...,synthetic
8,Technology Used,Augmented reality,**Company Name:** SecureNet Solutions \n**Web...,synthetic
9,Technology Used,Augmented reality,**Company Name:** Urban Mobility Systems \n**...,synthetic


In [1396]:
triplet_rows=[]
for i, group in positives_df.groupby(['aspect','query']):
    
    aspect=group['aspect'].iloc[0]
    query=group['query'].iloc[0]
    positives=group['factsheet'].to_list()
    
    for positive in positives:
        if 'Industry' in aspect:
            strategy_dict= {'rule':20,'venture':10,'synthetic':5}
        else:
            strategy_dict= {'venture':5,'synthetic':27}
        
        remaining=32
        negatives=[]
        i=0
        
        for strategy,number in strategy_dict.items():
            try:
                all_negatives=negatives_df_group.get_group((aspect,query,strategy))
            except KeyError:
                print(f'counld not find {(aspect,query,strategy)}')
                i+=1
                continue

            if i==len(strategy_dict)-1:
                sampled=all_negatives.sample(min(len(all_negatives),remaining))
                
            else:
                sampled=all_negatives.sample(min(len(all_negatives),strategy_dict[strategy]))
                
            remaining=remaining-len(sampled)
            
            _negatives=sampled.factsheet.to_list()
            negatives.extend(_negatives)
            i+=1
                
                
        _triplets=[{'query':(aspect,query),'positive':positive,'negative':negative} for negative in negatives]
        triplet_rows.extend(_triplets)
            
                

counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Direct sales', 'venture')
counld not find ('Business Model', 'Franchise', 'venture')
counld not find ('Business Model', 'Franchise', 'venture')
counld not find ('Business Model', 'Franchise', 'venture')
counld not find ('Business Model', 'Franchise', 'venture')
counld not find ('Business Model', 'Franchise', 'venture')
counld not find ('Business Model', 'Franchise', 'venture')
counld not find ('Business

In [1397]:
triplets_df=pd.DataFrame(triplet_rows)

In [1398]:
triplets_df

,query,positive,negative
0,"(Business Model, B2B)",* * the company holdings fact sheet * * \n\n *...,* * the company fact sheet * * \n\n * * compan...
1,"(Business Model, B2B)",* * the company holdings fact sheet * * \n\n *...,* * company name :* * colter ventures \n\n * *...
2,"(Business Model, B2B)",* * the company holdings fact sheet * * \n\n *...,* * company fact sheet : wy group * * \n\n * *...
3,"(Business Model, B2B)",* * the company holdings fact sheet * * \n\n *...,**Company Name:** TravelBuddy **Website:** tr...
4,"(Business Model, B2B)",* * the company holdings fact sheet * * \n\n *...,**Company Name:** SportShop **Website:** spor...
...,...,...,...
536421,"(Technology Used, Wearable technology)",* * actionstreamer company fact sheet * * \n\n...,**Company Name:** FinServe Innovations **Webs...
536422,"(Technology Used, Wearable technology)",* * actionstreamer company fact sheet * * \n\n...,**Company Name:** EduTech Solutions \n**Websi...
536423,"(Technology Used, Wearable technology)",* * actionstreamer company fact sheet * * \n\n...,**Company Name:** CleanTech Innovations \n**W...
536424,"(Technology Used, Wearable technology)",* * actionstreamer company fact sheet * * \n\n...,**Company Name:** AgriTech Innovations **Webs...


In [1399]:
import random
k=[1,2]
random.shuffle(k)

In [1400]:
k

[2, 1]

In [1401]:
split_strategy={'train':0.85,'val':0.15}

In [1402]:
triplets_group=triplets_df.groupby(['query'])
split_groups=[]
for _, group in triplets_group:
    
    split_={split:prop*len(group) for split,prop in split_strategy.items()}
    split_array=[]
    i=0
    poitives_df=group[['positive']].drop_duplicates()
    
    for split_key, prop in split_strategy.items():
        if i==len(split_strategy)-1:
            n=len(poitives_df)-len(split_array)
        else:
            n=int(prop*len(poitives_df))
        
        split_array.extend([split_key]*n)
        i+=1
    for s in range(5):
        random.shuffle(split_array)
    
    poitives_df['split']=split_array
    group=group.merge(poitives_df,how='left')
    split_groups.append(group)

In [1403]:
split_df=pd.concat(split_groups)

In [1404]:
# split_df

In [1409]:
split_df['split'].value_counts()

split
train    449908
val       86518
Name: count, dtype: int64

In [1406]:
83608/449619

0.1859529957586312

In [1407]:
# query_negatives_dict.keys()

In [1410]:
# split_df.split.

In [1411]:
split_df.to_pickle('../synthetic_data/split.pkl')

In [1412]:
train_triplets_df=split_df[split_df.split=='train']

In [1413]:
val_triplets_df=split_df[split_df.split=='val']

In [1414]:
# train_triplets_df['query']=train_triplets_df['query'].apply(lambda x:'||'.join(x))

In [1415]:
# val_triplets_df['query']=val_triplets_df['query'].apply(lambda x:'||'.join(x))

In [1428]:
train_collections=pd.concat([train_triplets_df[['positive']].rename(columns={'positive':'collection'}),train_triplets_df[['negative']].rename(columns={'negative':'collection'})]).drop_duplicates()

In [1429]:
val_collections=pd.concat([val_triplets_df[['positive']].rename(columns={'positive':'collection'}),val_triplets_df[['negative']].rename(columns={'negative':'collection'})]).drop_duplicates()

In [1430]:
train_collections['id']=list(range(len(train_collections)))
val_collections['id']=list(range(len(val_collections)))

In [1431]:
train_collections.to_pickle('../synthetic_data/collection.train.raw.pkl')
val_collections.to_pickle('../synthetic_data/collection.val.raw.pkl')

In [1432]:
train_queries=train_triplets_df[['query']].drop_duplicates()
val_queries=val_triplets_df[['query']].drop_duplicates()

In [1433]:
train_queries['id']=list(range(len(train_queries)))

In [1434]:
train_queries.to_pickle('../synthetic_data/queries.train.raw.pkl')

In [1435]:
val_queries['id']=list(range(len(val_queries)))

In [1436]:
val_queries.to_pickle('../synthetic_data/queries.val.raw.pkl')

In [1437]:
train_triplets_df=train_triplets_df.merge(train_queries[['query','id']].rename(columns={'id':'qid'}),how='left')
train_triplets_df=train_triplets_df.merge(train_collections[['collection','id']].rename(columns={'id':'pid','collection':'positive'}),how='left')
train_triplets_df=train_triplets_df.merge(train_collections[['collection','id']].rename(columns={'id':'nid','collection':'negative'}),how='left')

In [1438]:
### let's remove **Overall Assessment:** for synthetic negatives
train_triplets_df.loc[train_triplets_df.negative.isin(synthetic_negatives_df.factsheet),'negative']=train_triplets_df.loc[train_triplets_df.negative.isin(synthetic_negatives_df.factsheet),'negative'].apply(lambda x:
                                                                                                                                                                                                     x.split('**Overall Assessment:**')[0])


In [1439]:
# train_triplets_df.loc[train_triplets_df.negative.isin(synthetic_negatives_df.factsheet.apply(lambda x:x.split('**Overall Assessment:**')[0])),'negative'].iloc[0]

In [1440]:
val_triplets_df=val_triplets_df.merge(val_queries[['query','id']].rename(columns={'id':'qid'}),how='left')
val_triplets_df=val_triplets_df.merge(val_collections[['collection','id']].rename(columns={'id':'pid','collection':'positive'}),how='left')
val_triplets_df=val_triplets_df.merge(val_collections[['collection','id']].rename(columns={'id':'nid','collection':'negative'}),how='left')

In [1441]:
### let's remove **Overall Assessment:** for synthetic negatives
val_triplets_df.loc[val_triplets_df.negative.isin(synthetic_negatives_df.factsheet),'negative']=val_triplets_df.loc[val_triplets_df.negative.isin(synthetic_negatives_df.factsheet),'negative'].apply(lambda x:
                                                                                                                                                                                                     x.split('**Overall Assessment:**')[0])


In [1442]:
val_triplets_df.to_pickle('../synthetic_data/triples.val.raw.pkl')

In [1443]:

train_triplets_df.to_pickle('../synthetic_data/triples.train.raw.pkl')

In [1444]:
### creating NLQ variations of queries

def generate_nlq(query,n_queries):
    success=False
    response_format=NaturalLanguageQueryResponse
    
    sys_msg_path1='../prompts/query_diversification.txt'
    
    system_message1=open(sys_msg_path1,'r').read()
    
    llm1=OpenAIChatCompletion(model='gpt-4o',system_message=system_message1)
    
    if 'Industry' in query[0]:
        definitions='''- **Customer Industry**: This refers to the industry in which the company's customers operate. For example, if a company provides AI solutions to legal industry, the customer industry is the "Legal Industry.
        - **Industry**: This refers to the sector in which the company operates and creates solutions. For example, a company creating AI applications or LLMs is part of the "Technology and AI Industry.
        Please be careful, both are different as stated above. The queries should be such that the difference is very clear without any confusion\n\n
        '''
        prompt1=definitions+f"\nPlease provide {n_queries} natural language queries that are different ways an investor could ask for companies related to the query: {query}. Keep the Queries less that 10-12 words"
    else:
        prompt1=f"\nPlease provide {n_queries} natural language queries that are different ways an investor could ask for companies related to the query: {query}. Keep the Queries less that 10-12 words"
    temperature=0.5
    
    res1=llm1.structured_response(prompt1,temperature,response_format)

    try:
        r1=json.loads(res1)

        success=True
        return success,r1
    except Exception as e:
        
        print(f'failed for {query} - {e}')
        success=False
        return success,[]

In [804]:
res=generate_factsheets(('Busniess Model','B2B'),20)

In [806]:
train_queries['aspect']=train_queries['query'].apply(lambda x: x[0])

In [811]:
query_dict={}
for aspect,group in tqdm(train_queries.groupby('aspect')):
    
    queries=group['query'].to_list()
    
    args=[([(query[0],query[1]),20],{}) for query in queries]
    
    nlq_queries=parallel_execute(
                        func= generate_nlq,
                        argument_sets=args,
                        max_workers = 25,
                        show_progress= True,
                        desc = "generating negative factsheets"
                    )
    generate_nlq_=dict(zip(queries,nlq_queries))
    
    query_dict={**query_dict,**generate_nlq_}
    

100%|██████████| 7/7 [01:06<00:00,  9.50s/it]


In [814]:
len(query_dict.keys())

409

In [815]:
train_nlq_rows=[]
for query, nlqs in query_dict.items():
    train_nlq_rows.append({'query':query,'nlq':nlqs[1]})

In [829]:
train_nlq_df=pd.DataFrame(train_nlq_rows)

In [830]:
train_nlq_df.to_pickle('../synthetic_data/train_nlq_df.pkl')

In [822]:
val_queries['aspect']=val_queries['query'].apply(lambda x: x[0])

In [823]:
query_dict={}
for aspect,group in tqdm(val_queries.groupby('aspect')):
    
    queries=group['query'].to_list()
    
    args=[([(query[0],query[1]),10],{}) for query in queries]
    
    nlq_queries=parallel_execute(
                        func= generate_nlq,
                        argument_sets=args,
                        max_workers = 25,
                        show_progress= True,
                        desc = "generating negative factsheets"
                    )
    generate_nlq_=dict(zip(queries,nlq_queries))
    
    query_dict={**query_dict,**generate_nlq_}
    

100%|██████████| 7/7 [00:47<00:00,  6.75s/it]


In [824]:
val_nlq_rows=[]
for query, nlqs in query_dict.items():
    val_nlq_rows.append({'query':query,'nlq':nlqs[1]})

In [825]:
val_nlq_df=pd.DataFrame(val_nlq_rows)

In [827]:
val_nlq_df.to_pickle('../synthetic_data/val_nlq_df.pkl')

In [1446]:
train_triplets_df[train_triplets_df['query'].apply(lambda x:'Technology' in x[0])].iloc[2000:2050]

,query,positive,negative,split,qid,pid,nid
417444,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** EduTech Innovations \n**Web...,train,380,11972,28548
417445,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** RoboTech Systems **Website:...,train,380,11972,28581
417446,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** Urban Mobility Systems \n**...,train,380,11972,28552
417447,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** CloudSync Solutions \n**Web...,train,380,11972,28559
417448,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** AeroDynamics Corp **Website...,train,380,11972,28549
417449,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** CloudWave Systems **Website...,train,380,11972,28574
417450,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** MedTech Solutions \n**Websi...,train,380,11972,28584
417451,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** AutoTech Innovations **Webs...,train,380,11972,28561
417452,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** CleanEnergy Solutions \n**W...,train,380,11972,28596
417453,"(Technology Used, 5G technology)",* * astrome company fact sheet * * \n\n * * 1 ...,**Company Name:** MedTech Solutions **Website...,train,380,11972,28600


In [1445]:
print(train_triplets_df[train_triplets_df['query'].apply(lambda x:'Technology' in x[0])].iloc[2010].negative)

**Company Name:** SkyNet Solutions  **Website:** skynetsolutions.com  **Industry:** Satellite Communication  **Target Audience:** Global enterprises requiring satellite data services  **Products & Services:**  - SkyConnect: Satellite-based communication network  - SkyData: High-speed satellite internet service  **Unique Selling Propositions (USPs) & Key Differentiators:**  - Global coverage with low latency  - Robust network resilience in remote areas  **Business Model:** B2B Subscription  **Technology Used:** Utilizes satellite communication technologies and advanced signal processing. Their infrastructure is based on geostationary and low-earth orbit satellites.  **Revenue Model:** Subscription-based with tiered pricing for bandwidth usage  **Competitive Landscape:** Competes with HughesNet, Viasat, and other satellite internet providers  


In [1450]:
# train_triplets_df

In [1452]:
val_triplets_df

,query,positive,negative,split,qid,pid,nid
0,"(Business Model, B2B)",* * company fact sheet : rsi ngd group of comp...,* * company name :* * colter ventures \n\n * *...,val,0,0,2604
1,"(Business Model, B2B)",* * company fact sheet : rsi ngd group of comp...,* * the company fact sheet * * \n\n * * compan...,val,0,0,2605
2,"(Business Model, B2B)",* * company fact sheet : rsi ngd group of comp...,* * company fact sheet : wy group * * \n\n * *...,val,0,0,2606
3,"(Business Model, B2B)",* * company fact sheet : rsi ngd group of comp...,**Company Name:** PetCare **Website:** petcar...,val,0,0,2607
4,"(Business Model, B2B)",* * company fact sheet : rsi ngd group of comp...,**Company Name:** Appetito **Website:** appet...,val,0,0,2608
...,...,...,...,...,...,...,...
86513,"(Technology Used, Wearable technology)",* * company fact sheet : ftsame * * \n\n * * 1...,**Company Name:** AgriTech Innovations \n**We...,val,408,2603,20011
86514,"(Technology Used, Wearable technology)",* * company fact sheet : ftsame * * \n\n * * 1...,**Company Name:** TravelTech Innovations \n**...,val,408,2603,19989
86515,"(Technology Used, Wearable technology)",* * company fact sheet : ftsame * * \n\n * * 1...,**Company Name:** UrbanTech Innovations \n**W...,val,408,2603,20007
86516,"(Technology Used, Wearable technology)",* * company fact sheet : ftsame * * \n\n * * 1...,**Company Name:** RetailTech Solutions \n**We...,val,408,2603,20010
